In [1]:
"""
Exploratory data figures for the resting-state / phenotype dataset.

Produces a small suite of colorful-but-professional figures to show the spread of
every variable and the rough relationships among them. If the specparam output
exists, aperiodic figures are added too.

Inputs
  participants.tsv                     (age, education, HC group)
  <phenotype>/bdi.tsv upps.tsv lifestyle.tsv hc_usage.tsv
  derivatives/preproc/specparam/aperiodic_per_subject_channel.csv   (optional)

Outputs (derivatives/preproc/figures/)
  fig1_sample_overview.png
  fig2_score_distributions.png
  fig3_correlation_heatmap.png
  fig4_group_comparisons.png
  fig5_hc_lifestyle_breakdown.png
  fig6_aperiodic_overview.png          (only if specparam CSV present)
  phenotype_merged.csv                 (tidy, decoded table used for the plots)

Requires: pandas numpy matplotlib seaborn
"""

import os
import numpy as np
import pandas as pd
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import seaborn as sns

# ============================================================
# CONFIG + STYLE
# ============================================================
BIDS_ROOT     = "/Users/elizabethkaplan/Desktop/ds007615"
DERIV_ROOT    = os.path.join(BIDS_ROOT, "derivatives", "preproc")
PHENOTYPE_DIR = "/Users/elizabethkaplan/Desktop/phenotype"
PARTICIPANTS  = os.path.join(BIDS_ROOT, "participants.tsv")
APERIODIC_CSV = os.path.join(DERIV_ROOT, "specparam", "aperiodic_per_subject_channel.csv")
OUT_DIR       = os.path.join(DERIV_ROOT, "figures")
os.makedirs(OUT_DIR, exist_ok=True)

sns.set_theme(style="whitegrid", context="talk")
plt.rcParams.update({"figure.dpi": 120, "savefig.dpi": 140, "axes.titleweight": "bold",
                     "axes.spines.top": False, "axes.spines.right": False})
PAL = ["#2A9D8F", "#E76F51", "#E9C46A", "#264653", "#8AB17D", "#F4A261", "#9B5DE5"]
GROUP_PAL = {"Current HC": "#2A9D8F", "Past HC": "#E9C46A", "Never": "#E76F51"}


def save(fig, name):
    fig.tight_layout()
    fig.savefig(os.path.join(OUT_DIR, name), bbox_inches="tight")
    plt.close(fig)
    print("saved", name)


# ============================================================
# LOAD + MERGE + DECODE
# ============================================================
def sid(df):
    df = df.copy()
    df["subject"] = df["participant_id"].str.replace("sub-", "", regex=False)
    return df.set_index("subject")

parts = sid(pd.read_csv(PARTICIPANTS, sep="\t"))
bdi   = sid(pd.read_csv(os.path.join(PHENOTYPE_DIR, "bdi.tsv"), sep="\t"))
upps  = sid(pd.read_csv(os.path.join(PHENOTYPE_DIR, "upps.tsv"), sep="\t"))
life  = sid(pd.read_csv(os.path.join(PHENOTYPE_DIR, "lifestyle.tsv"), sep="\t"))
hc    = sid(pd.read_csv(os.path.join(PHENOTYPE_DIR, "hc_usage.tsv"), sep="\t"))

df = parts.join([bdi.drop(columns="participant_id"),
                 upps.drop(columns="participant_id"),
                 life.drop(columns="participant_id"),
                 hc.drop(columns="participant_id")], how="left")

# decode categoricals to readable labels
df["HC group"] = df["group"].map({1: "Current HC", 2: "Past HC", 3: "Never"})
df["Education"] = df["edu"].map({1: "Primary", 2: "Secondary (voc)",
                                 3: "Secondary (acad)", 4: "University"})
df["HC type"] = df["main_prev"].map({1: "Progestin-only", 2: "Combined"})
df["Delivery"] = df["oc_nonoc"].map({1: "Oral", 2: "Non-oral"})
df["Menstrual phase"] = df["mens_phase"].map({1: "Follicular", 2: "Ovulatory", 3: "Luteal"})
df["Daily nicotine"] = df["daily_nicotine"].map({1: "Yes", 2: "No"})
df["Recreational drugs"] = df["drugs_bin"].map({1: "Yes", 2: "No"})
df["Medication use"] = df["medication_use"].map({1: "Yes", 2: "No"})

# optional aperiodic: per-subject mean across channels, per condition
if os.path.exists(APERIODIC_CSV):
    ap = pd.read_csv(APERIODIC_CSV, dtype={"subject": str})
    for acq in ["ec", "eo"]:
        sub = ap[ap["acq"] == acq].groupby("subject")[["exponent", "offset"]].mean()
        df[f"exponent_{acq}"] = sub["exponent"]
        df[f"offset_{acq}"] = sub["offset"]
    HAS_AP = "exponent_ec" in df.columns and df["exponent_ec"].notna().any()
else:
    HAS_AP = False

df.to_csv(os.path.join(OUT_DIR, "phenotype_merged.csv"))
GROUP_ORDER = ["Current HC", "Past HC", "Never"]


# ============================================================
# FIG 1 — sample overview
# ============================================================
fig, ax = plt.subplots(1, 3, figsize=(16, 4.5))
sns.countplot(data=df, x="HC group", order=GROUP_ORDER, hue="HC group",
              palette=GROUP_PAL, legend=False, ax=ax[0])
ax[0].set_title("Hormonal contraceptive status"); ax[0].set_xlabel("")
sns.histplot(data=df, x="age", bins=14, kde=True, color=PAL[0], ax=ax[1])
ax[1].set_title("Age distribution"); ax[1].set_xlabel("age (years)")
edu_order = ["Primary", "Secondary (voc)", "Secondary (acad)", "University"]
sns.countplot(data=df, y="Education", order=[e for e in edu_order if e in df["Education"].values],
              hue="Education", palette="crest", legend=False, ax=ax[2])
ax[2].set_title("Education"); ax[2].set_ylabel("")
fig.suptitle("Sample overview  (N = %d)" % len(df), fontsize=18, fontweight="bold")
save(fig, "fig1_sample_overview.png")


# ============================================================
# FIG 2 — distributions of continuous scores
# ============================================================
score_vars = [("bdi_total", "BDI-II total"), ("bdi_cog", "BDI cognitive"),
              ("bdi_som", "BDI somatic"), ("upps_urgency", "UPPS urgency"),
              ("upps_premeditation", "UPPS (lack) premeditation"),
              ("upps_perseverance", "UPPS (lack) perseverance"),
              ("upps_sensation_seeking", "UPPS sensation seeking"),
              ("alcohol_units", "Alcohol (units/wk)"), ("prev_duration", "HC duration (yrs)")]
fig, axes = plt.subplots(3, 3, figsize=(15, 12))
for ax, (col, lab), color in zip(axes.ravel(), score_vars, PAL * 2):
    d = df[col].dropna()
    sns.histplot(d, bins=14, kde=True, color=color, ax=ax)
    ax.axvline(d.mean(), color="0.25", ls="--", lw=1.5)
    ax.set_title(lab); ax.set_xlabel("")
fig.suptitle("Distributions of questionnaire & lifestyle scores  (dashed = mean)",
             fontsize=18, fontweight="bold")
save(fig, "fig2_score_distributions.png")


# ============================================================
# FIG 3 — correlation heatmap among continuous variables
# ============================================================
corr_vars = ["age", "bdi_total", "bdi_cog", "bdi_som", "upps_urgency",
             "upps_premeditation", "upps_perseverance", "upps_sensation_seeking",
             "alcohol_units", "prev_duration", "cycle_length"]
if HAS_AP:
    corr_vars += ["exponent_ec", "offset_ec", "exponent_eo", "offset_eo"]
corr_vars = [c for c in corr_vars if c in df.columns]
corr = df[corr_vars].corr()
mask = np.triu(np.ones_like(corr, dtype=bool), k=1)
fig, ax = plt.subplots(figsize=(1.0 * len(corr_vars) + 2, 1.0 * len(corr_vars)))
sns.heatmap(corr, mask=mask, annot=True, fmt=".2f", cmap="vlag", center=0,
            vmin=-1, vmax=1, square=True, linewidths=.5,
            cbar_kws={"shrink": .7, "label": "Pearson r"}, annot_kws={"size": 9}, ax=ax)
ax.set_title("Correlations among continuous measures", fontweight="bold")
save(fig, "fig3_correlation_heatmap.png")


# ============================================================
# FIG 4 — key measures by HC group
# ============================================================
comp_vars = [("age", "Age"), ("bdi_total", "BDI-II total"),
             ("upps_urgency", "UPPS urgency"), ("upps_sensation_seeking", "UPPS sensation seeking")]
fig, axes = plt.subplots(1, 4, figsize=(18, 4.6))
for ax, (col, lab) in zip(axes, comp_vars):
    sns.violinplot(data=df, x="HC group", y=col, order=GROUP_ORDER, hue="HC group",
                   palette=GROUP_PAL, legend=False, inner="quartile", cut=0, ax=ax)
    sns.stripplot(data=df, x="HC group", y=col, order=GROUP_ORDER, color="0.2",
                  size=3, alpha=.5, ax=ax)
    ax.set_title(lab); ax.set_xlabel(""); ax.set_ylabel("")
fig.suptitle("Measures by hormonal-contraceptive status", fontsize=18, fontweight="bold")
save(fig, "fig4_group_comparisons.png")


# ============================================================
# FIG 5 — HC method & lifestyle breakdowns
# ============================================================
cat_vars = [("HC type", "crest"), ("Delivery", "crest"), ("Menstrual phase", "flare"),
            ("Daily nicotine", "mako"), ("Recreational drugs", "mako"), ("Medication use", "mako")]
fig, axes = plt.subplots(2, 3, figsize=(15, 8.5))
for ax, (col, cmap) in zip(axes.ravel(), cat_vars):
    d = df[col].dropna()
    order = d.value_counts().index.tolist()
    sns.countplot(x=d, order=order, hue=d, palette=cmap, legend=False, ax=ax)
    ax.set_title(col); ax.set_xlabel(""); ax.set_ylabel("count")
    for p in ax.patches:
        ax.annotate(int(p.get_height()), (p.get_x() + p.get_width() / 2, p.get_height()),
                    ha="center", va="bottom", fontsize=11)
fig.suptitle("Contraceptive method & lifestyle composition", fontsize=18, fontweight="bold")
save(fig, "fig5_hc_lifestyle_breakdown.png")


# ============================================================
# FIG 6 — aperiodic overview (only if specparam ran)
# ============================================================
if HAS_AP:
    fig, ax = plt.subplots(1, 3, figsize=(17, 5))
    sns.regplot(data=df, x="age", y="exponent_ec", color=PAL[0],
                scatter_kws=dict(s=45, alpha=.7), ax=ax[0])
    ax[0].set_title("Aperiodic exponent vs age (EC)")
    ax[0].set_xlabel("age (years)"); ax[0].set_ylabel("exponent")
    sns.violinplot(data=df, x="HC group", y="exponent_ec", order=GROUP_ORDER, hue="HC group",
                   palette=GROUP_PAL, legend=False, inner="quartile", cut=0, ax=ax[1])
    sns.stripplot(data=df, x="HC group", y="exponent_ec", order=GROUP_ORDER,
                  color="0.2", size=3, alpha=.5, ax=ax[1])
    ax[1].set_title("Exponent by HC status (EC)"); ax[1].set_xlabel(""); ax[1].set_ylabel("exponent")
    lim = [min(df["exponent_ec"].min(), df["exponent_eo"].min()),
           max(df["exponent_ec"].max(), df["exponent_eo"].max())]
    sns.scatterplot(data=df, x="exponent_ec", y="exponent_eo", hue="HC group",
                    palette=GROUP_PAL, s=55, ax=ax[2])
    ax[2].plot(lim, lim, "k--", lw=1)
    ax[2].set_title("EC vs EO exponent"); ax[2].set_xlabel("eyes-closed"); ax[2].set_ylabel("eyes-open")
    ax[2].legend(title="", fontsize=9)
    fig.suptitle("Aperiodic parameter overview", fontsize=18, fontweight="bold")
    save(fig, "fig6_aperiodic_overview.png")
else:
    print("aperiodic CSV not found — skipped fig6 (run specparam_analysis.py first)")

print(f"\nAll figures -> {OUT_DIR}")

saved fig1_sample_overview.png
saved fig2_score_distributions.png
saved fig3_correlation_heatmap.png
saved fig4_group_comparisons.png
saved fig5_hc_lifestyle_breakdown.png
aperiodic CSV not found — skipped fig6 (run specparam_analysis.py first)

All figures -> /Users/elizabethkaplan/Desktop/ds007615/derivatives/preproc/figures
